## 1. Setup and Data Loading

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

### Download Dataset from Kaggle

**Note:** To download the dataset, you need to:
1. Install kaggle: `pip install kaggle`
2. Set up Kaggle API credentials:
   - Go to Kaggle → Account → API → Create New API Token
   - Place `kaggle.json` in `~/.kaggle/`
   - Run: `chmod 600 ~/.kaggle/kaggle.json`

Alternatively, you can manually download from: https://www.kaggle.com/datasets/brandao/diabetes

In [ ]:
import os

# Create data directory if it doesn't exist
data_dir = '../data/raw'
os.makedirs(data_dir, exist_ok=True)

# Download dataset using Kaggle API
# Uncomment the following lines to download:
# !kaggle datasets download -d brandao/diabetes -p {data_dir} --unzip

print(f"Data directory created: {data_dir}")
print("Please download the dataset manually or uncomment the kaggle command above.")

In [ ]:
# Load the dataset
# Note: The dataset filename may vary - adjust accordingly
data_path = '../data/raw/diabetic_data.csv'

# Try to load the data
try:
    df = pd.read_csv(data_path)
    print(f"Dataset loaded successfully!")
    print(f"Shape: {df.shape}")
except FileNotFoundError:
    print(f"Dataset not found at {data_path}")
    print("Please download the dataset first.")
    df = None

## 2. Initial Data Exploration

In [ ]:
if df is not None:
    # Display basic information
    print("Dataset Shape:")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")
    print("\n" + "="*80 + "\n")
    
    # Display first few rows
    print("First 5 rows:")
    display(df.head())
    
    print("\n" + "="*80 + "\n")
    
    # Display last few rows
    print("Last 5 rows:")
    display(df.tail())

In [ ]:
if df is not None:
    # Data types and non-null counts
    print("Data Types and Non-Null Counts:")
    print(df.info())

In [ ]:
if df is not None:
    # Statistical summary for numerical features
    print("Statistical Summary (Numerical Features):")
    display(df.describe())

In [ ]:
if df is not None:
    # Statistical summary for categorical features
    print("Statistical Summary (Categorical Features):")
    categorical_cols = df.select_dtypes(include=['object']).columns
    if len(categorical_cols) > 0:
        display(df[categorical_cols].describe())
    else:
        print("No categorical columns found.")

## 3. Exploratory Data Analysis (EDA)

### 3.1 Missing Values Analysis

In [ ]:
if df is not None:
    # Calculate missing values
    missing_data = pd.DataFrame({
        'Column': df.columns,
        'Missing_Count': df.isnull().sum(),
        'Missing_Percentage': (df.isnull().sum() / len(df)) * 100,
        'Dtype': df.dtypes
    })
    missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values(
        'Missing_Percentage', ascending=False
    )
    
    if len(missing_data) > 0:
        print(f"Total columns with missing values: {len(missing_data)}")
        display(missing_data)
        
        # Visualize missing values
        plt.figure(figsize=(12, 6))
        plt.bar(missing_data['Column'], missing_data['Missing_Percentage'])
        plt.xticks(rotation=90)
        plt.xlabel('Columns')
        plt.ylabel('Missing Percentage (%)')
        plt.title('Missing Values by Column')
        plt.tight_layout()
        plt.show()
    else:
        print("No missing values found in the dataset!")

### 3.2 Target Variable Analysis

Note: The target variable for readmission prediction needs to be identified from the dataset.
Common names: 'readmitted', 'readmission', 'target', etc.

In [ ]:
if df is not None:
    # Identify potential target variable
    # Common column names for readmission
    potential_targets = ['readmitted', 'readmission', 'target', 'outcome']
    
    target_col = None
    for col in potential_targets:
        if col in df.columns:
            target_col = col
            break
    
    if target_col is None:
        # List columns to help identify target
        print("Could not automatically identify target column.")
        print("\nAll columns in dataset:")
        for i, col in enumerate(df.columns, 1):
            print(f"{i}. {col}")
        print("\nPlease identify the target variable manually.")
    else:
        print(f"Target variable identified: {target_col}")
        print(f"\nTarget variable distribution:")
        print(df[target_col].value_counts())
        print(f"\nTarget variable distribution (%):")
        print(df[target_col].value_counts(normalize=True) * 100)
        
        # Visualize target distribution
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Count plot
        df[target_col].value_counts().plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
        axes[0].set_title(f'Distribution of {target_col}')
        axes[0].set_xlabel(target_col)
        axes[0].set_ylabel('Count')
        axes[0].tick_params(axis='x', rotation=45)
        
        # Pie chart
        df[target_col].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
        axes[1].set_title(f'Proportion of {target_col}')
        axes[1].set_ylabel('')
        
        plt.tight_layout()
        plt.show()

### 3.3 Numerical Features Analysis

In [ ]:
if df is not None:
    # Identify numerical columns
    numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    print(f"Number of numerical features: {len(numerical_cols)}")
    print(f"Numerical features: {numerical_cols}")

In [ ]:
if df is not None and len(numerical_cols) > 0:
    # Distribution plots for numerical features
    n_features = min(len(numerical_cols), 12)  # Limit to first 12 features
    n_cols = 3
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
    
    for idx, col in enumerate(numerical_cols[:n_features]):
        axes[idx].hist(df[col].dropna(), bins=30, edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'Distribution of {col}')
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Frequency')
    
    # Hide unused subplots
    for idx in range(n_features, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()

In [ ]:
if df is not None and len(numerical_cols) > 0:
    # Box plots to identify outliers
    n_features = min(len(numerical_cols), 12)
    n_cols = 3
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
    
    for idx, col in enumerate(numerical_cols[:n_features]):
        axes[idx].boxplot(df[col].dropna())
        axes[idx].set_title(f'Box Plot of {col}')
        axes[idx].set_ylabel(col)
    
    # Hide unused subplots
    for idx in range(n_features, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()

### 3.4 Categorical Features Analysis

In [ ]:
if df is not None:
    # Identify categorical columns
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    print(f"Number of categorical features: {len(categorical_cols)}")
    print(f"Categorical features: {categorical_cols}")
    
    # Display unique values for each categorical feature
    if len(categorical_cols) > 0:
        print("\nUnique values per categorical feature:")
        for col in categorical_cols:
            n_unique = df[col].nunique()
            print(f"  {col}: {n_unique} unique values")
            if n_unique <= 10:
                print(f"    Values: {df[col].unique().tolist()}")

In [ ]:
if df is not None and len(categorical_cols) > 0:
    # Visualize categorical features with low cardinality
    low_cardinality_cols = [col for col in categorical_cols if df[col].nunique() <= 10]
    
    if len(low_cardinality_cols) > 0:
        n_features = min(len(low_cardinality_cols), 9)
        n_cols = 3
        n_rows = (n_features + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
        axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
        
        for idx, col in enumerate(low_cardinality_cols[:n_features]):
            df[col].value_counts().plot(kind='bar', ax=axes[idx], color='lightcoral', edgecolor='black')
            axes[idx].set_title(f'Distribution of {col}')
            axes[idx].set_xlabel(col)
            axes[idx].set_ylabel('Count')
            axes[idx].tick_params(axis='x', rotation=45)
        
        # Hide unused subplots
        for idx in range(n_features, len(axes)):
            axes[idx].set_visible(False)
        
        plt.tight_layout()
        plt.show()

### 3.5 Correlation Analysis

In [ ]:
if df is not None and len(numerical_cols) > 1:
    # Calculate correlation matrix
    correlation_matrix = df[numerical_cols].corr()
    
    # Plot correlation heatmap
    plt.figure(figsize=(14, 10))
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, linewidths=1)
    plt.title('Correlation Matrix of Numerical Features')
    plt.tight_layout()
    plt.show()
    
    # Identify highly correlated features
    print("\nHighly correlated feature pairs (|correlation| > 0.7):")
    high_corr = []
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            if abs(correlation_matrix.iloc[i, j]) > 0.7:
                high_corr.append((
                    correlation_matrix.columns[i],
                    correlation_matrix.columns[j],
                    correlation_matrix.iloc[i, j]
                ))
    
    if high_corr:
        for feat1, feat2, corr_val in high_corr:
            print(f"  {feat1} <-> {feat2}: {corr_val:.3f}")
    else:
        print("  No highly correlated pairs found.")

## 4. Data Preprocessing

### 4.1 Handle Special Values

The diabetes dataset often contains '?' as missing value indicator.

In [ ]:
if df is not None:
    # Create a copy for preprocessing
    df_processed = df.copy()
    
    # Replace '?' with NaN
    df_processed.replace('?', np.nan, inplace=True)
    
    # Check for any other special values that might represent missing data
    special_values = ['Unknown', 'unknown', 'UNKNOWN', 'None', 'NA', 'N/A', '']
    for val in special_values:
        df_processed.replace(val, np.nan, inplace=True)
    
    print("Special values replaced with NaN.")
    print(f"\nMissing values after replacement:")
    print(df_processed.isnull().sum()[df_processed.isnull().sum() > 0])

### 4.2 Handle Missing Values

In [ ]:
if df is not None:
    # Strategy for handling missing values
    print("Missing Value Handling Strategy:")
    print("-" * 50)
    
    missing_threshold = 0.5  # Drop columns with >50% missing
    
    # Identify columns to drop
    cols_to_drop = []
    for col in df_processed.columns:
        missing_pct = df_processed[col].isnull().sum() / len(df_processed)
        if missing_pct > missing_threshold:
            cols_to_drop.append(col)
            print(f"  DROP: {col} ({missing_pct*100:.1f}% missing)")
    
    # Drop high-missing columns
    if cols_to_drop:
        df_processed.drop(columns=cols_to_drop, inplace=True)
        print(f"\nDropped {len(cols_to_drop)} columns with >{missing_threshold*100}% missing values.")
    
    # Handle remaining missing values
    print("\nImputation strategy for remaining columns:")
    for col in df_processed.columns:
        if df_processed[col].isnull().sum() > 0:
            if df_processed[col].dtype in ['int64', 'float64']:
                # Numerical: impute with median
                median_val = df_processed[col].median()
                df_processed[col].fillna(median_val, inplace=True)
                print(f"  {col}: Imputed with median ({median_val})")
            else:
                # Categorical: impute with mode
                mode_val = df_processed[col].mode()[0] if not df_processed[col].mode().empty else 'Unknown'
                df_processed[col].fillna(mode_val, inplace=True)
                print(f"  {col}: Imputed with mode ('{mode_val}')")
    
    print(f"\nFinal missing values: {df_processed.isnull().sum().sum()}")

### 4.3 Data Type Conversions

In [ ]:
if df is not None:
    # Convert data types as needed
    print("Data type conversions:")
    print("-" * 50)
    
    # Identify columns that should be numeric but are stored as objects
    for col in df_processed.select_dtypes(include=['object']).columns:
        try:
            # Attempt to convert to numeric
            df_processed[col] = pd.to_numeric(df_processed[col])
            print(f"  {col}: Converted to numeric")
        except (ValueError, TypeError):
            # Keep as categorical
            pass
    
    print("\nFinal data types:")
    print(df_processed.dtypes.value_counts())

### 4.4 Handle Duplicates

In [ ]:
if df is not None:
    # Check for duplicates
    n_duplicates = df_processed.duplicated().sum()
    print(f"Number of duplicate rows: {n_duplicates}")
    
    if n_duplicates > 0:
        print(f"Removing {n_duplicates} duplicate rows...")
        df_processed.drop_duplicates(inplace=True)
        print(f"New shape: {df_processed.shape}")
    else:
        print("No duplicates found.")

## 5. Feature Engineering

### 5.1 Create Age Groups

In [ ]:
if df is not None:
    # Check if age column exists
    age_cols = [col for col in df_processed.columns if 'age' in col.lower()]
    
    if age_cols:
        age_col = age_cols[0]
        print(f"Creating age groups from '{age_col}' column...")
        
        # Check if age is already in bins (like '[0-10)', '[10-20)', etc.)
        sample_val = df_processed[age_col].iloc[0]
        if isinstance(sample_val, str) and '[' in str(sample_val):
            print(f"  {age_col} is already in categorical bins.")
            df_processed['age_group'] = df_processed[age_col]
        else:
            # Create age groups from numeric age
            df_processed['age_group'] = pd.cut(
                df_processed[age_col],
                bins=[0, 18, 30, 45, 60, 75, 120],
                labels=['0-18', '19-30', '31-45', '46-60', '61-75', '76+']
            )
        
        print("\nAge group distribution:")
        print(df_processed['age_group'].value_counts().sort_index())
    else:
        print("No age column found for age group creation.")

### 5.2 Create BMI Categories (if applicable)

In [ ]:
if df is not None:
    # Check for weight and height columns to calculate BMI
    weight_cols = [col for col in df_processed.columns if 'weight' in col.lower()]
    height_cols = [col for col in df_processed.columns if 'height' in col.lower()]
    
    if weight_cols and height_cols:
        weight_col = weight_cols[0]
        height_col = height_cols[0]
        
        # Calculate BMI (assuming weight in kg and height in meters)
        df_processed['bmi'] = df_processed[weight_col] / (df_processed[height_col] ** 2)
        
        # Create BMI categories
        df_processed['bmi_category'] = pd.cut(
            df_processed['bmi'],
            bins=[0, 18.5, 25, 30, 40, 100],
            labels=['Underweight', 'Normal', 'Overweight', 'Obese', 'Severely Obese']
        )
        
        print("BMI categories created.")
        print("\nBMI category distribution:")
        print(df_processed['bmi_category'].value_counts())
    else:
        print("Weight and/or height columns not found for BMI calculation.")

### 5.3 Create Medical History Features

In [ ]:
if df is not None:
    # Count number of diagnoses (if diagnosis columns exist)
    diag_cols = [col for col in df_processed.columns if 'diag' in col.lower()]
    
    if diag_cols:
        print(f"Found {len(diag_cols)} diagnosis columns: {diag_cols}")
        
        # Count non-null diagnoses
        df_processed['num_diagnoses'] = df_processed[diag_cols].notna().sum(axis=1)
        
        print("\nNumber of diagnoses distribution:")
        print(df_processed['num_diagnoses'].value_counts().sort_index())
    
    # Count number of medications (if medication columns exist)
    med_change_cols = [col for col in df_processed.columns if 'change' in col.lower()]
    
    if med_change_cols:
        df_processed['medication_changes'] = (df_processed[med_change_cols] == 'Ch').sum(axis=1)
        print(f"\nMedication changes feature created.")
    
    # Count number of procedures (if procedure columns exist)
    proc_cols = [col for col in df_processed.columns if 'num_procedures' in col.lower()]
    if proc_cols:
        print(f"Procedure count column found: {proc_cols[0]}")

### 5.4 Create Interaction Features

In [ ]:
if df is not None:
    # Create interaction features based on domain knowledge
    
    # Example: Interaction between time in hospital and number of procedures
    time_cols = [col for col in df_processed.columns if 'time_in_hospital' in col.lower()]
    proc_cols = [col for col in df_processed.columns if 'num_procedures' in col.lower()]
    
    if time_cols and proc_cols:
        time_col = time_cols[0]
        proc_col = proc_cols[0]
        
        df_processed['procedures_per_day'] = df_processed[proc_col] / (df_processed[time_col] + 1)
        print("Created 'procedures_per_day' interaction feature.")
    
    # Example: Interaction between number of medications and number of diagnoses
    med_cols = [col for col in df_processed.columns if 'num_medications' in col.lower()]
    
    if med_cols and diag_cols:
        med_col = med_cols[0]
        df_processed['meds_per_diagnosis'] = df_processed[med_col] / (df_processed['num_diagnoses'] + 1)
        print("Created 'meds_per_diagnosis' interaction feature.")

### 5.5 Encode Categorical Variables

In [ ]:
from sklearn.preprocessing import LabelEncoder

if df is not None:
    # Identify categorical columns (excluding target)
    categorical_cols = df_processed.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # Remove target column from encoding (if identified)
    if target_col and target_col in categorical_cols:
        categorical_cols.remove(target_col)
    
    print(f"Encoding {len(categorical_cols)} categorical features...")
    
    # Store label encoders for later use
    label_encoders = {}
    
    for col in categorical_cols:
        n_unique = df_processed[col].nunique()
        
        if n_unique <= 2:
            # Binary encoding
            le = LabelEncoder()
            df_processed[f'{col}_encoded'] = le.fit_transform(df_processed[col].astype(str))
            label_encoders[col] = le
            print(f"  {col}: Binary encoded ({n_unique} categories)")
        elif n_unique <= 10:
            # One-hot encoding for low cardinality
            dummies = pd.get_dummies(df_processed[col], prefix=col, drop_first=True)
            df_processed = pd.concat([df_processed, dummies], axis=1)
            print(f"  {col}: One-hot encoded ({n_unique} categories -> {len(dummies.columns)} features)")
        else:
            # Label encoding for high cardinality
            le = LabelEncoder()
            df_processed[f'{col}_encoded'] = le.fit_transform(df_processed[col].astype(str))
            label_encoders[col] = le
            print(f"  {col}: Label encoded ({n_unique} categories)")
    
    print(f"\nTotal features after encoding: {df_processed.shape[1]}")

## 6. Feature Selection

### 6.1 Remove Low Variance Features

In [ ]:
from sklearn.feature_selection import VarianceThreshold

if df is not None:
    # Select only numerical columns for variance threshold
    numerical_features = df_processed.select_dtypes(include=['int64', 'float64']).columns.tolist()
    
    if target_col and target_col in numerical_features:
        numerical_features.remove(target_col)
    
    print(f"Analyzing variance for {len(numerical_features)} numerical features...")
    
    # Apply variance threshold
    selector = VarianceThreshold(threshold=0.01)
    selector.fit(df_processed[numerical_features].fillna(0))
    
    # Identify low variance features
    low_variance_features = [feat for feat, var in zip(numerical_features, selector.variances_) if var < 0.01]
    
    if low_variance_features:
        print(f"\nFound {len(low_variance_features)} low variance features:")
        for feat in low_variance_features:
            print(f"  - {feat}")
        
        # Optionally remove low variance features
        # df_processed.drop(columns=low_variance_features, inplace=True)
        print("\nNote: Low variance features identified but not removed yet.")
    else:
        print("\nNo low variance features found.")

### 6.2 Feature Importance Analysis (Requires Target Variable)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

if df is not None and target_col:
    print("Calculating feature importance using Random Forest...")
    
    # Prepare features and target
    # Select only numerical features for this analysis
    feature_cols = df_processed.select_dtypes(include=['int64', 'float64']).columns.tolist()
    
    if target_col in feature_cols:
        feature_cols.remove(target_col)
    
    # Handle missing values
    X = df_processed[feature_cols].fillna(0)
    y = df_processed[target_col]
    
    # Encode target if categorical
    if y.dtype == 'object':
        le_target = LabelEncoder()
        y = le_target.fit_transform(y)
    
    # Train a simple Random Forest
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=10)
    rf.fit(X_train, y_train)
    
    # Get feature importances
    feature_importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': rf.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\nTop 20 Most Important Features:")
    display(feature_importance.head(20))
    
    # Visualize top features
    plt.figure(figsize=(10, 8))
    top_n = 20
    plt.barh(feature_importance['feature'][:top_n][::-1], 
             feature_importance['importance'][:top_n][::-1])
    plt.xlabel('Importance')
    plt.title(f'Top {top_n} Feature Importances')
    plt.tight_layout()
    plt.show()
    
elif df is not None:
    print("Target variable not identified. Skipping feature importance analysis.")
    print("Please identify the target variable manually to perform this analysis.")

### 6.3 Correlation-Based Feature Selection

In [ ]:
if df is not None:
    # Remove highly correlated features (multicollinearity)
    numerical_features = df_processed.select_dtypes(include=['int64', 'float64']).columns.tolist()
    
    if target_col and target_col in numerical_features:
        numerical_features.remove(target_col)
    
    if len(numerical_features) > 1:
        # Calculate correlation matrix
        corr_matrix = df_processed[numerical_features].corr().abs()
        
        # Find pairs of highly correlated features
        upper_triangle = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        
        # Find features with correlation > 0.9
        to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.9)]
        
        if to_drop:
            print(f"Found {len(to_drop)} highly correlated features (correlation > 0.9):")
            for feat in to_drop:
                print(f"  - {feat}")
            
            # Optionally drop these features
            # df_processed.drop(columns=to_drop, inplace=True)
            print("\nNote: Highly correlated features identified but not removed yet.")
        else:
            print("No highly correlated features found (threshold: 0.9).")

## 7. Save Preprocessed Data

In [ ]:
if df is not None:
    # Create processed data directory
    processed_dir = '../data/processed'
    os.makedirs(processed_dir, exist_ok=True)
    
    # Save processed dataframe
    output_path = os.path.join(processed_dir, 'processed_data.csv')
    df_processed.to_csv(output_path, index=False)
    
    print(f"Preprocessed data saved to: {output_path}")
    print(f"Final shape: {df_processed.shape}")
    
    # Save label encoders for future use
    import pickle
    encoder_path = os.path.join(processed_dir, 'label_encoders.pkl')
    with open(encoder_path, 'wb') as f:
        pickle.dump(label_encoders, f)
    print(f"Label encoders saved to: {encoder_path}")

## 8. Summary

In [ ]:
if df is not None:
    print("="*80)
    print("DATA PREPROCESSING SUMMARY")
    print("="*80)
    print(f"\nOriginal dataset shape: {df.shape}")
    print(f"Processed dataset shape: {df_processed.shape}")
    print(f"\nFeatures added: {df_processed.shape[1] - df.shape[1]}")
    print(f"\nMissing values remaining: {df_processed.isnull().sum().sum()}")
    print(f"\nData types:")
    print(df_processed.dtypes.value_counts())
    
    if target_col:
        print(f"\nTarget variable: {target_col}")
        print(f"Target distribution:")
        print(df_processed[target_col].value_counts(normalize=True) * 100)
    
    print("\n" + "="*80)
    print("Ready for model development!")
    print("="*80)